# Bola de padel — treinar o TEU detetor + cortar tempo util
Caminho **gratuito**: descarregas o dataset gratuito de bola de padel, treinas um YOLO (GPU gratis do Colab), e usas os **teus** pesos para detetar a bola no teu video -> regras de rally (deteccao_rallies v9) -> video condensado.

**Corre so depois** do `colab_ball_test.ipynb` confirmar que a bola se deteta. `Runtime -> T4 GPU`.

## 1. Instalar + montar Drive

In [ ]:
!pip install -q ultralytics roboflow opencv-python-headless
from google.colab import drive; drive.mount('/content/drive')
VIDEO_PATH='/content/drive/MyDrive/PadelPro_Vision/videos/Analise Padel Modelo - Parada 4 min.mp4'
import os; print('video?', os.path.exists(VIDEO_PATH))

## 2. Descarregar o dataset (gratis)
No Roboflow Universe, abre um dataset de bola de padel (ex.: `padel-ll7pp/padel-dataset`), carrega em **Download Dataset -> YOLOv8 -> show download code**, e **cola aqui o snippet** que te dao (ja traz a tua API key e a versao certa). Ele cria um objeto `dataset`.

In [ ]:
from roboflow import Roboflow
# >>> COLA AQUI o snippet do Roboflow (Download -> YOLOv8). Exemplo do formato: <<<
# rf = Roboflow(api_key='A_TUA_KEY')
# project = rf.workspace('padel-ll7pp').project('padel-dataset')
# dataset = project.version(1).download('yolov8')
print('dataset em:', dataset.location)

## 3. Treinar o YOLO (so a bola)
~20-30 min numa T4. Imagens a 960 para a bola pequena. Guarda os pesos no Drive.

In [ ]:
from ultralytics import YOLO
m=YOLO('yolov8n.pt')
m.train(data=f'{dataset.location}/data.yaml', epochs=40, imgsz=960, batch=8, project='/content/ball', name='padel_ball')
BEST='/content/ball/padel_ball/weights/best.pt'
import shutil,os; os.makedirs('/content/drive/MyDrive/PadelPro_Vision/weights/ball_yolo',exist_ok=True)
shutil.copy(BEST,'/content/drive/MyDrive/PadelPro_Vision/weights/ball_yolo/best.pt'); print('pesos guardados no Drive')

## 4. Detetar a bola no video -> bola_visivel por frame

In [ ]:
import cv2, numpy as np
model=YOLO(BEST)
cap=cv2.VideoCapture(VIDEO_PATH); fps=cap.get(cv2.CAP_PROP_FPS); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
bola=[False]*n; i=0
while True:
    ok,fr=cap.read()
    if not ok: break
    r=model.predict(fr, conf=0.15, imgsz=960, verbose=False)[0]
    bola[i]= len(r.boxes)>0
    i+=1
cap.release()
print(f'bola visivel em {sum(bola)}/{n} frames ({100*sum(bola)/n:.0f}%)')
import json; json.dump(bola, open('/content/drive/MyDrive/PadelPro_Vision/jogos/parada4_teste/bola_visivel.json','w'))

## 5. Segmentar rallies (regras v9 — deteccao_rallies)

In [ ]:
def frame_para_ts(f,fps): s=f/fps; mm=int(s//60); return f'{mm:02d}:{s-mm*60:06.3f}'
def segmentar(bola, fps, gap_fora_s=2.0, pre_serve_s=1.5, pos_fim_s=2.5, min_rally_s=1.0):
    n=len(bola); gap=int(gap_fora_s*fps); pre=int(pre_serve_s*fps); pos=int(pos_fim_s*fps); ml=int(min_rally_s*fps)
    runs=[]; i=0
    while i<n:
        if not bola[i]: i+=1; continue
        ini=i
        while i+1<n and bola[i+1]: i+=1
        runs.append([ini,i]); i+=1
    blocos=[]
    for run in runs:
        if blocos and (run[0]-blocos[-1][1]-1)<=gap: blocos[-1][1]=run[1]
        else: blocos.append(run)
    segs=[]
    for a,b in blocos:
        if b-a+1>=ml: segs.append([max(0,a-pre), min(n-1,b+pos)])
    for k in range(len(segs)-1):
        if segs[k][1]>=segs[k+1][0]:
            mid=(segs[k][1]+segs[k+1][0])//2; segs[k][1]=mid; segs[k+1][0]=mid+1
    return segs
segs=segmentar(bola,fps)
dur=[(b-a)/fps for a,b in segs]
print(f'{len(segs)} rallies | tempo util {sum(dur):.0f}s/{n/fps:.0f}s ({100*sum(dur)/(n/fps):.0f}%) | media {np.mean(dur):.1f}s')

## 6. Cortar o video condensado

In [ ]:
import subprocess, os
os.makedirs('/content/clips',exist_ok=True); parts=[]
for j,(a,b) in enumerate(segs):
    o=f'/content/clips/r{j:02d}.mp4'
    subprocess.run(['ffmpeg','-y','-ss',f'{a/fps:.2f}','-to',f'{b/fps:.2f}','-i',VIDEO_PATH,'-c:v','libx264','-crf','23','-preset','veryfast','-an',o],capture_output=True)
    parts.append(os.path.abspath(o))
open('/content/list.txt','w').write('\n'.join(f"file '{p}'" for p in parts))
OUT='/content/drive/MyDrive/PadelPro_Vision/jogos/parada4_teste/TempoUtil_bola.mp4'
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','/content/list.txt','-c','copy',OUT],capture_output=True)
print('condensado guardado no Drive:', os.path.exists(OUT))